In [5]:
import requests
from bs4 import BeautifulSoup
import html, json
import time
from config import vnexpress_rss_feeds, dantri_rss_feeds, kenh14_rss_feeds, afamily_rss_feeds, cafebiz_rss_feeds

### VnExpress Crawling

In [ ]:
def vnexpress_crawling():
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7',
        'Referer': 'https://vnexpress.net/',
        'Connection': 'keep-alive',
    }
    valid_news_data = []
    url_existed = set()
    # Duyệt qua từng chủ đề và link rss của mỗi chủ đề 
    for category, rss_url in vnexpress_rss_feeds.items():
        # Gửi yêu cầu lấy dữ liệu trong rss
        response = requests.get(rss_url, timeout=20)
        if response.status_code != 200: 
            continue
        xml_soup = BeautifulSoup(response.content, 'xml')
        # Lấy ra tất cả bài báo trong chủ đề (rss) đó
        items = xml_soup.find_all('item')
        news_data = []
        # Duyệt qua từng bài báo trong chủ đề đó để lấy được title và url trong rss
        for item in items:
            # Lấy ra title
            title = item.title.text.strip()
            # Lấy ra url
            url = item.find('link').text.strip()
            if url in url_existed:
                continue
            url_existed.add(url)
            news_data.append({
                'title': title,
                'url': url
            })
        # Duyệt qua từng bài báo đã lưu đó để xử lý
        for item in news_data:
            url = item['url']
            if not url:
                continue
            time.sleep(0.5)
            response = requests.get(url, headers=headers, timeout=20)
            if response.status_code == 200:
                soup = BeautifulSoup(response.content, 'html.parser')
                # Tìm vùng chứa nội dung chính
                article_div = soup.find('div', class_=['container detail-new'])
                if not article_div:
                    continue
                # Tạo list lưu các object text, img để xây dựng web về sau
                content_block = []
                # Tạo list để lưu nội dung các text của từng bài báo để train 
                text_list = []
                author = None
                # Duyệt qua từng element trong vùng chứa chính có thẻ p và picture
                for element in article_div.find_all(['p', 'picture']):
                    # Kiểm tra nếu đây là thẻ p
                    if element.name == 'p':
                        # Nếu thẻ p có text bắt đầu bằng `>>` và `Ảnh` -> bỏ qua
                        if element.text.strip().startswith(('>>', 'Ảnh')):
                            continue
                        # Nếu thẻ p có chứa thẻ strong (đây là thẻ chứa tên tác giả) -> bỏ qua
                        element_author = element.find('strong')
                        if element_author:
                            # Gán cho biến author
                            author = element_author.text.strip()
                            continue
                        # Tìm các class có trong thẻ p
                        element_classes = element.get('class', [])
                        # Nếu thẻ p có class muốn lấy, lấy ra text rồi lưu vào
                        if 'description' in element_classes or 'Normal' in element_classes:
                            text = element.text.strip()
                            if text:
                                content_block.append({
                                    'type': 'text',
                                    'content': text
                                })
                                text_list.append(text)
                    # Kiểm tra nếu đây là picture 
                    if element.name == 'picture':
                        # Lấy ra vùng chứa img
                        img_tag = element.find('img')
                        if img_tag:
                            img_url = img_tag['data-src'].strip()
                            img_caption = img_tag['alt'].strip()
                            content_block.append({
                                'type': 'image',
                                'url': img_url,
                                'caption': img_caption
                            })
                # Nối các phần tử trong text_list lại bằng `. `
                texts = '. '.join(text_list)
                # Lấy ra ngày đăng
                published_at = article_div.find('span', class_='date').text.strip()
                # Cập nhật dữ liệu cho mỗi object
                item['texts'] = texts
                item['author'] = author
                item['category'] = category
                item['published_at'] = published_at
                item['source'] = 'VnExpress'
                item['content_block'] = content_block
                # Thêm những bài báo có DOM hợp lệ vào
                valid_news_data.append(item)
    return valid_news_data

In [3]:
def save_to_json(data, file_path='output.jsonl'):
    with open(file_path, mode='a', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

In [49]:
save_to_json(vnexpress_crawling())

### Dân Trí Crawling

In [ ]:
def dantri_crawling():
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7',
        'Referer': 'https://dantri.com.vn/',
        'Connection': 'keep-alive',
    }
    valid_news_data = []
    url_existed = set()
    # Duyệt qua từng chủ đề và link rss của mỗi chủ đề 
    for category, rss_url in dantri_rss_feeds.items():
        # Gửi yêu cầu lấy dữ liệu trong rss
        response = requests.get(rss_url, timeout=20)
        if response.status_code != 200: 
            continue
        xml_soup = BeautifulSoup(response.content, 'xml')
        # Lấy ra tất cả bài báo trong chủ đề (rss) đó
        items = xml_soup.find_all('item')
        news_data = []
        # Duyệt qua từng bài báo trong chủ đề đó để lấy được title và url trong rss
        for item in items:
            # Lấy ra title
            title = item.title.text
            # Lấy ra url
            url = item.find('link').text
            if url in url_existed:
                continue
            url_existed.add(url)
            news_data.append({
                'title': title,
                'url': url
            })
        # Duyệt qua từng bài báo đã lưu đó để xử lý
        for item in news_data:
            url = item['url']
            if not url:
                continue
            time.sleep(0.5)
            response = requests.get(url, headers=headers, timeout=20)
            if response.status_code == 200:
                soup = BeautifulSoup(response.content, 'html.parser')
                # Tìm vùng chứa nội dung chính
                article_div = soup.find('article')
                if not article_div:
                    continue
                # Tạo list lưu các object text, img để xây dựng web về sau
                content_block = []
                # Tạo list để lưu nội dung các text của từng bài báo để train 
                text_list = []
                # Duyệt qua từng element trong vùng chứa chính có thẻ p và picture
                for element in article_div.find_all(['h2', 'p', 'figure']):
                    # Kiểm tra nếu đây là thẻ h2 hoặc p
                    if element.name == 'h2' or element.name == 'p':
                        if element.find_parent('figcaption'):
                            continue
                        text = element.text.strip()
                        if text:
                            content_block.append({
                                'type': 'text',
                                'content': text
                            })
                            text_list.append(text)
                    # Kiểm tra nếu đây là figure 
                    if element.name == 'figure':
                        # Lấy ra vùng chứa img
                        img_tag = element.find('img')
                        if img_tag:
                            img_url = img_tag['data-src'].strip()
                            img_caption = ""
                            caption_tag = element.find('figcaption')
                            if caption_tag:
                                img_caption = caption_tag.text.strip()
                            content_block.append({
                                'type': 'image',
                                'url': img_url,
                                'caption': img_caption
                            })
                # Nối các phần tử trong text_list lại bằng `. `
                texts = '. '.join(text_list)
                # Lấy ra tác giả
                author = None
                a_tag = article_div.find('a')
                if a_tag:
                    author = a_tag.text.strip()
                # Lấy ra ngày đăng
                published_at = article_div.find('time').text.strip()
                # Cập nhật dữ liệu cho mỗi object
                item['texts'] = texts
                item['author'] = author
                item['category'] = category
                item['published_at'] = published_at
                item['source'] = 'Dân Trí'
                item['content_block'] = content_block
                # Thêm những bài báo có DOM hợp lệ vào
                valid_news_data.append(item)
    return valid_news_data

In [56]:
save_to_json(dantri_crawling())

### Kênh 14 Crawling

In [6]:
def kenh14_crawling():
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7',
        'Referer': 'https://kenh14.vn/',
        'Connection': 'keep-alive',
    }
    valid_news_data = []
    url_existed = set()
    # Duyệt qua từng chủ đề và link rss của mỗi chủ đề 
    for category, rss_url in kenh14_rss_feeds.items():
        # Gửi yêu cầu lấy dữ liệu trong rss
        response = requests.get(rss_url, timeout=20)
        if response.status_code != 200: 
            continue
        xml_soup = BeautifulSoup(response.content, 'xml')
        # Lấy ra tất cả bài báo trong chủ đề (rss) đó
        items = xml_soup.find_all('item')
        news_data = []
        # Duyệt qua từng bài báo trong chủ đề đó để lấy được title và url trong rss
        for item in items:
            # Lấy ra title
            title = item.title.text
            # Lấy ra url
            url = item.find('link').text
            if url in url_existed:
                continue
            url_existed.add(url)
            news_data.append({
                'title': title,
                'url': url
            })
        # Duyệt qua từng bài báo đã lưu đó để xử lý
        for item in news_data:
            url = item['url']
            if not url:
                continue
            time.sleep(0.5)
            response = requests.get(url, headers=headers, timeout=20)
            if response.status_code == 200:
                soup = BeautifulSoup(response.content, 'html.parser')
                # Tìm vùng chứa nội dung chính
                article_div = soup.find('div', class_='klw-new-content')
                if not article_div:
                    continue
                # Tạo list lưu các object text, img để xây dựng web về sau
                content_block = []
                # Tạo list để lưu nội dung các text của từng bài báo để train 
                text_list = []
                # Duyệt qua từng element trong vùng chứa chính có thẻ p và picture
                for element in article_div.find_all(['h2', 'p', 'figure']):
                    # Kiểm tra nếu đây là thẻ h2 hoặc p
                    if element.name == 'h2' or element.name == 'p':
                        if element.find_parent('figcaption'):
                            continue
                        text = element.text.strip()
                        if text:
                            content_block.append({
                                'type': 'text',
                                'content': text
                            })
                            text_list.append(text)
                    # Kiểm tra nếu đây là figure 
                    if element.name == 'figure':
                        # Lấy ra vùng chứa img
                        img_tag = element.find('img')
                        if img_tag:
                            img_url = img_tag['src'].strip()
                            img_caption = ""
                            caption_tag = element.find('figcaption')
                            if caption_tag:
                                img_caption = caption_tag.text.strip()
                            content_block.append({
                                'type': 'image',
                                'url': img_url,
                                'caption': img_caption
                            })
                # Nối các phần tử trong text_list lại bằng `. `
                texts = '. '.join(text_list)
                # Lấy ra tác giả
                author = None
                author_tag = soup.find('span', class_='kbwcm-author')
                if author_tag:
                    author = author_tag.text.strip()
                    author = author[:-1]
                # Lấy ra ngày đăng
                published_at = soup.find('span', class_='kbwcm-time').text.strip()
                # Cập nhật dữ liệu cho mỗi object
                item['texts'] = texts
                item['author'] = author
                item['category'] = category
                item['published_at'] = published_at
                item['source'] = 'Kênh 14'
                item['content_block'] = content_block
                # Thêm những bài báo có DOM hợp lệ vào
                valid_news_data.append(item)
    return valid_news_data

In [7]:
save_to_json(kenh14_crawling())

### Afamily Crawling

In [8]:
def afamily_crawling():
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7',
        'Referer': 'https://afamily.vn/',
        'Connection': 'keep-alive',
    }
    valid_news_data = []
    url_existed = set()
    # Duyệt qua từng chủ đề và link rss của mỗi chủ đề 
    for category, rss_url in afamily_rss_feeds.items():
        # Gửi yêu cầu lấy dữ liệu trong rss
        response = requests.get(rss_url, timeout=20)
        if response.status_code != 200: 
            continue
        xml_soup = BeautifulSoup(response.content, 'xml')
        # Lấy ra tất cả bài báo trong chủ đề (rss) đó
        items = xml_soup.find_all('item')
        news_data = []
        # Duyệt qua từng bài báo trong chủ đề đó để lấy được title và url trong rss
        for item in items:
            # Lấy ra title
            title = item.title.text
            # Lấy ra url
            url = item.find('link').text
            if url in url_existed:
                continue
            url_existed.add(url)
            news_data.append({
                'title': title,
                'url': url
            })
        # Duyệt qua từng bài báo đã lưu đó để xử lý
        for item in news_data:
            url = item['url']
            if not url:
                continue
            time.sleep(0.5)
            response = requests.get(url, headers=headers, timeout=20)
            if response.status_code == 200:
                soup = BeautifulSoup(response.content, 'html.parser')
                # Tìm vùng chứa nội dung chính
                article_div = soup.find('article')
                if not article_div:
                    continue
                # Tạo list lưu các object text, img để xây dựng web về sau
                content_block = []
                # Tạo list để lưu nội dung các text của từng bài báo để train 
                text_list = []
                # Duyệt qua từng element trong vùng chứa chính có thẻ p và picture
                for element in article_div.find_all(['h2', 'p', 'figure']):
                    # Kiểm tra nếu đây là thẻ h2 hoặc p
                    if element.name == 'h2' or element.name == 'p':
                        if element.find_parent('figcaption'):
                            continue
                        text = element.text.strip()
                        if text:
                            content_block.append({
                                'type': 'text',
                                'content': text
                            })
                            text_list.append(text)
                    # Kiểm tra nếu đây là figure 
                    if element.name == 'figure':
                        # Lấy ra vùng chứa img
                        img_tag = element.find('img')
                        if img_tag:
                            img_url = img_tag['src'].strip()
                            img_caption = ""
                            caption_tag = element.find('figcaption')
                            if caption_tag:
                                img_caption = caption_tag.text.strip()
                            content_block.append({
                                'type': 'image',
                                'url': img_url,
                                'caption': img_caption
                            })
                # Nối các phần tử trong text_list lại bằng `. `
                texts = '. '.join(text_list)
                # Lấy ra tác giả
                author = None
                author_tag = soup.find('span', class_='afcba-name')
                if author_tag:
                    author = author_tag.text.strip()
                    author = author[:-1]
                # Lấy ra ngày đăng
                published_at = soup.find('span', class_='afcba-time').text.strip()
                # Cập nhật dữ liệu cho mỗi object
                item['texts'] = texts
                item['author'] = author
                item['category'] = category
                item['published_at'] = published_at
                item['source'] = 'Afamily'
                item['content_block'] = content_block
                # Thêm những bài báo có DOM hợp lệ vào
                valid_news_data.append(item)
    return valid_news_data

In [9]:
save_to_json(afamily_crawling())

### CafeBiz Crawling

In [14]:
def cafebiz_crawling():
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7',
        'Referer': 'https://cafebiz.vn/',
        'Connection': 'keep-alive',
    }
    valid_news_data = []
    url_existed = set()
    # Duyệt qua từng chủ đề và link rss của mỗi chủ đề 
    for category, rss_url in cafebiz_rss_feeds.items():
        # Gửi yêu cầu lấy dữ liệu trong rss
        response = requests.get(rss_url, timeout=40)
        if response.status_code != 200: 
            continue
        xml_soup = BeautifulSoup(response.content, 'xml')
        # Lấy ra tất cả bài báo trong chủ đề (rss) đó
        items = xml_soup.find_all('item')
        news_data = []
        # Duyệt qua từng bài báo trong chủ đề đó để lấy được title và url trong rss
        for item in items:
            # Lấy ra title
            title = item.title.text
            # Lấy ra url
            url = item.find('link').text
            if url in url_existed:
                continue
            url_existed.add(url)
            news_data.append({
                'title': title,
                'url': url
            })
        # Duyệt qua từng bài báo đã lưu đó để xử lý
        for item in news_data:
            url = item['url']
            if not url:
                continue
            time.sleep(0.5)
            response = requests.get(url, headers=headers, timeout=40)
            if response.status_code == 200:
                soup = BeautifulSoup(response.content, 'html.parser')
                # Tìm vùng chứa nội dung chính
                article_div = soup.find('div', class_='content')
                if not article_div:
                    continue
                # Tạo list lưu các object text, img để xây dựng web về sau
                content_block = []
                # Tạo list để lưu nội dung các text của từng bài báo để train 
                text_list = []
                # Duyệt qua từng element trong vùng chứa chính có thẻ p và picture
                for element in article_div.find_all(['h2', 'p', 'figure']):
                    # Kiểm tra nếu đây là thẻ h2 hoặc p
                    if element.name == 'h2' or element.name == 'p':
                        if element.find_parent('figcaption'):
                            continue
                        if element.find('strong'):
                            continue
                        text = element.text.strip()
                        if text:
                            content_block.append({
                                'type': 'text',
                                'content': text
                            })
                            text_list.append(text)
                    # Kiểm tra nếu đây là figure 
                    if element.name == 'figure':
                        # Lấy ra vùng chứa img
                        img_tag = element.find('img')
                        if img_tag:
                            img_url = img_tag['src'].strip()
                            img_caption = ""
                            caption_tag = element.find('figcaption')
                            if caption_tag:
                                img_caption = caption_tag.text.strip()
                            content_block.append({
                                'type': 'image',
                                'url': img_url,
                                'caption': img_caption
                            })
                # Nối các phần tử trong text_list lại bằng `. `
                texts = '. '.join(text_list)
                # Lấy ra tác giả
                author = None
                author_tag = soup.find('span', class_='author')
                if author_tag:
                    author = author_tag.text.strip()
                    author = author[:-2]
                # Lấy ra ngày đăng
                published_at = soup.find('span', class_='time').text.strip()
                # Cập nhật dữ liệu cho mỗi object
                item['texts'] = texts
                item['author'] = author
                item['category'] = category
                item['published_at'] = published_at
                item['source'] = 'CafeBiz'
                item['content_block'] = content_block
                # Thêm những bài báo có DOM hợp lệ vào
                valid_news_data.append(item)
    return valid_news_data

In [15]:
save_to_json(cafebiz_crawling())

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


In [17]:
import pandas as pd
df = pd.read_json('output.jsonl', lines=True)
df

,title,url,texts,author,category,published_at,source,content_block
0,Lao động trung niên tìm việc nhiều nhưng thị t...,https://vnexpress.net/lao-dong-trung-nien-tim-...,Gần 40% số người tìm việc thuộc nhóm trung niê...,Lê Tuyết,Thời sự,"Thứ tư, 8/4/2026, 00:00 (GMT+7)",VnExpress,"[{'type': 'text', 'content': 'Gần 40% số người..."
1,Tổng Bí thư Tô Lâm gặp mặt cán bộ Văn phòng Ch...,https://vnexpress.net/tong-bi-thu-to-lam-gap-m...,Sau khi được Quốc hội bầu giữ chức Chủ tịch nư...,Vũ Tuân,Thời sự,"Thứ ba, 7/4/2026, 22:40 (GMT+7)",VnExpress,"[{'type': 'text', 'content': 'Sau khi được Quố..."
2,Nhiệt độ nhiều nơi vượt 40 độ C,https://vnexpress.net/nhiet-do-nhieu-noi-vuot-...,Cả nước bước vào ngày thứ hai của đợt nắng nón...,Gia Chính,Thời sự,"Thứ ba, 7/4/2026, 20:49 (GMT+7)",VnExpress,"[{'type': 'text', 'content': 'Cả nước bước vào..."
3,Các ủy ban của Quốc hội kiện toàn nhân sự,https://vnexpress.net/cac-uy-ban-cua-quoc-hoi-...,"Ngày 7/4, Hội đồng Dân tộc và một số Ủy ban củ...",Sơn Hà,Thời sự,"Thứ ba, 7/4/2026, 20:30 (GMT+7)",VnExpress,"[{'type': 'text', 'content': 'Ngày 7/4, Hội đồ..."
4,Cháy nhà 4 tầng ở trung tâm TP HCM,https://vnexpress.net/chay-nha-4-tang-o-trung-...,Khói lửa bùng lên từ lầu một ngôi nhà gần ngã ...,Đình Văn,Thời sự,"Thứ ba, 7/4/2026, 17:45 (GMT+7)",VnExpress,"[{'type': 'text', 'content': 'Khói lửa bùng lê..."
...,...,...,...,...,...,...,...,...
2295,"Cận Vía Thần Tài, một Tiệm vàng lớn tuyên bố c...",https://cafebiz.vn/can-ngay-via-than-tai-mot-t...,Tiệm vàng với 34 cơ sở trên toàn quốc khẳng đị...,Phan Trang,BizMoney,24/02/2026 10:59 AM,CafeBiz,"[{'type': 'text', 'content': 'Tiệm vàng với 34..."
2296,Giá vàng sáng 24/2 vượt 5.200 USD/ounce,https://cafebiz.vn/gia-vang-sang-24-2-vuot-520...,"Chỉ trong một ngày, giá vàng đã có mức tăng mạ...",Huyền Băng,BizMoney,24/02/2026 07:10 AM,CafeBiz,"[{'type': 'text', 'content': 'Chỉ trong một ng..."
2297,Giá vàng thế giới chạm mốc 5.200 USD/ounce,https://cafebiz.vn/gia-vang-the-gioi-cham-moc-...,Thị trường vàng ngày càng nóng lên.. Trong 24 ...,Băng Băng,BizMoney,23/02/2026 22:25 PM,CafeBiz,"[{'type': 'text', 'content': 'Thị trường vàng ..."
2298,"Trước ngày vía Thần Tài: Không chỉ vàng, giá b...",https://cafebiz.vn/truoc-ngay-via-than-tai-kho...,"Tại thị trường trong nước, giá bạc cũng đã tăn...",Ngọc Tú,BizMoney,23/02/2026 10:20 AM,CafeBiz,"[{'type': 'text', 'content': 'Tại thị trường t..."


In [18]:
df.to_csv('/Users/qtumy/Desktop/news_crawling/data.csv', index=False, encoding='utf-8-sig')